In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))
import numpy as np
import matplotlib.pyplot as plt
from Train_fun import train_fun
from scipy.interpolate import Rbf
from RK import solve_rk4_adaptive_fixed_dt
from RK import build_f_sparse_mixed
from Bulid_Library import build_polynomial_library
from SSSR import SSSR
from SSSR import linear_reg

In [2]:
def rrmse(x,y):
    return (np.mean((x-y)**2))**0.5/(np.mean(y**2))**0.5

# Data generation

In [3]:
# define initial condition
Lx, Ly = 20.0, 20.0      
Nx, Ny = 65, 65
x = np.linspace(-Lx/2, Lx/2, Nx)
y = np.linspace(-Ly/2, Ly/2, Ny)
X, Y = np.meshgrid(x, y, indexing='ij')

xc, yc = 0.0, 0.0
s = np.hypot(X - xc, Y - yc)         
theta = np.arctan2(Y - yc, X - xc)  


delta,alpha = 1, 1                 
A = np.tanh((s / delta)**alpha)
k1, k2, phi0  = 0.2, 0, 0                  
theta = theta + k1 * s + k2 * s**2 + phi0

u0 = A * np.sin(theta)
v0 = A * np.cos(theta)

In [21]:
def RD_data(u0,v0,alpha,Lx,Nx,Ly,Ny,dt,T_final):
    dx = Lx / (Nx - 1)
    dy = Ly / (Ny - 1)
    
    # 时间参数
    Nt = int(T_final / dt) + 1
    U_ori = np.zeros((Nt,Nx*Ny))
    V_ori = np.zeros((Nt,Nx*Ny))
    
    # 构造空间网格
    x = np.linspace(0, Lx, Nx)
    y = np.linspace(0, Ly, Ny)
    X, Y = np.meshgrid(x, y, indexing='ij')
    
    # 初始条件 I：
    np.random.seed(0)
    u = u0
    v = v0
    
    # 定义计算二维 Laplacian 的函数，采用中心差分，并在边界上简单处理 Neumann 零通量条件
    def laplacian(Z, dx, dy):
        # 内部点使用 np.roll 进行周期性差分（后面边界单独处理）
        Zxx = (np.roll(Z, -1, axis=0) - 2 * Z + np.roll(Z, 1, axis=0)) / dx**2
        Zyy = (np.roll(Z, -1, axis=1) - 2 * Z + np.roll(Z, 1, axis=1)) / dy**2
        LZ = Zxx + Zyy
        
        # 对边界采用 Neumann 边界条件（简单一阶外推）
        # 上下边界
        LZ[0, :]    = (Z[1, :] - Z[0, :]) / dx**2 + (np.roll(Z, -1, axis=1)[0, :] - 2*Z[0, :] + np.roll(Z, 1, axis=1)[0, :]) / dy**2
        LZ[-1, :]   = (Z[-2, :] - Z[-1, :]) / dx**2 + (np.roll(Z, -1, axis=1)[-1, :] - 2*Z[-1, :] + np.roll(Z, 1, axis=1)[-1, :]) / dy**2
        # 左右边界
        LZ[:, 0]    = (np.roll(Z, -1, axis=0)[:, 0] - 2*Z[:, 0] + np.roll(Z, 1, axis=0)[:, 0]) / dx**2 + (Z[:, 1] - Z[:, 0]) / dy**2
        LZ[:, -1]   = (np.roll(Z, -1, axis=0)[:, -1] - 2*Z[:, -1] + np.roll(Z, 1, axis=0)[:, -1]) / dx**2 + (Z[:, -2] - Z[:, -1]) / dy**2
        
        return LZ
    
    # 时间推进（采用显式欧拉法）
    for n in range(Nt):
        # 计算扩散项（Laplacian）
        Lu = laplacian(u, dx, dy)
        Lv = laplacian(v, dx, dy)
        
        # 根据给定的反应扩散方程构造反应项：
        # u_t = alpha*(u_xx + u_yy) - u*v^2 - u^3 + v^3 + u^2*v + u
        # v_t = alpha*(v_xx + v_yy) + v - u*v^2 - u^3 - v^3 - u^2*v
        react_u = - u * (v**2) - u**3 + (v**3) + u**2 * v + u
        react_v = v - u*(v**2) - u**3 - (v**3) - u**2 * v
        
        # 显式欧拉更新
        u_new = u + dt * (alpha * Lu + react_u)
        v_new = v + dt * (alpha * Lv + react_v)
        U_ori[n,:] = np.squeeze(u.reshape(-1,1))
        V_ori[n,:] = np.squeeze(v.reshape(-1,1))
        
        # 更新解
        u, v = u_new, v_new
        
    return U_ori,V_ori

In [88]:
m = 401
Para_number = 15
Para = np.linspace(0.2, 0.7, Para_number)
U_all = []

for point in Para:
    U_alpha,V_alpha = RD_data(u0,v0,point,Lx,Nx,Ly,Ny,0.001,4)
    U_alpha = U_alpha[::10,:]
    U_all.append(U_alpha)

U = np.vstack(U_all)  

# POD

In [89]:
U_mean = np.mean(U, axis=0)
U_centered = U - U_mean
U_snapshots = U_centered.T  
Phi, Sigma, Vt = np.linalg.svd(U_snapshots, full_matrices=False)

latent_dim = 7
Phi_r = Phi[:,:latent_dim]
Z = np.transpose(np.dot(Phi_r.T, U_snapshots))  # shape = (m, r)
U_reconstructed = np.dot(Z, Phi_r.T) + U_mean
print('reconstruct error:', rrmse(U_reconstructed,U))

reconstruct error: 0.0015294282181830623


# SSSR

In [90]:
dt = 4/(m-1)
dZ = np.zeros_like(Z)

for i in range(Para_number):
    start = i * m
    end = (i + 1) * m
    a_seg = Z[start:end, :]  # shape: (m, r)
    
    da_seg = np.zeros_like(a_seg)
    da_seg[1:-1] = (a_seg[2:] - a_seg[:-2]) / (2 * dt)
    da_seg[0] = (a_seg[1] - a_seg[0]) / dt
    da_seg[-1] = (a_seg[-1] - a_seg[-2]) / dt
    
    dZ[start:end, :] = da_seg

In [91]:
include_functions = False
include_bias=False
degree = 2
Theta, feature_names = build_polynomial_library(Z, degree=degree, include_bias=include_bias, include_functions=include_functions)
sparsity_level = 3
Z_pred = Z.copy()
coef_matrix = np.zeros((Para_number,(sparsity_level+1)*latent_dim))
Supports = np.zeros((latent_dim,sparsity_level))

for k in range(latent_dim):
    loss = 0
    support = SSSR(Theta, dZ[:,k].reshape(-1,1), sparsity_level=sparsity_level, Para_number=Para_number)
    print('Support set for latent variable {}:'.format(k), support)
    Supports[k,:] = support
    Library = []
    for j in range(Para_number):
        Target = dZ[m*j:m*(j+1),k].reshape(-1,1)
        state = Z[m*j:m*(j+1),k].reshape(-1,1)
        Feature = Theta[m*j:m*(j+1),support]
        reg, pred = linear_reg(Feature,Target)
        coef_matrix[j,(sparsity_level+1)*k] = reg.intercept_
        coef_matrix[j,(sparsity_level+1)*k+1:(sparsity_level+1)*(k+1)] = reg.coef_
        loss = loss + rrmse(pred,Target)
        state_next_pred = state + dt * pred
        state_pred = state.copy()
        state_pred[1:] = state_next_pred[:-1]
        Library.append(state_pred)
    print('Regression error of latent variable {}:'.format(k), loss/Para_number)
    state_pred = np.vstack(Library)
    Z_pred[:,k] = np.squeeze(state_pred)
    Supports = Supports.astype('int')

Support set for latent variable 0: [1, 4, 3]
Regression error of latent variable 0: 0.0006782678233612226
Support set for latent variable 1: [0, 1, 2]
Regression error of latent variable 1: 0.0022299565721972084
Support set for latent variable 2: [1, 2, 3]
Regression error of latent variable 2: 0.03894693310697965
Support set for latent variable 3: [2, 0, 1]
Regression error of latent variable 3: 0.02638814838685295
Support set for latent variable 4: [1, 8, 3]
Regression error of latent variable 4: 0.03950357376585687
Support set for latent variable 5: [8, 4, 22]
Regression error of latent variable 5: 0.15419786698115256
Support set for latent variable 6: [9, 17, 18]
Regression error of latent variable 6: 0.2665329523199719


In [92]:
Library = []
for j in range(Para_number):
    f, info = build_f_sparse_mixed(feature_names , Supports=Supports, 
                              coef_matrix=coef_matrix[j].reshape(latent_dim,-1), latent_dim=latent_dim)
    z0 =  Z[m*j,:]
    T, Y = solve_rk4_adaptive_fixed_dt(f, 0, dt*(m-1), z0, dt, rtol=1e-6, atol=1e-9, h0=1e-2, h_min=0.00001)
    Library.append(Y)
Z_pred_multi_step = np.vstack(Library)

In [93]:
print('The prediction error of Z by multiple step:', rrmse(Z_pred_multi_step,Z))
U_pred_multi_step = np.dot(Z_pred_multi_step, Phi_r.T) + U_mean  # shape = (m, n)
print('The prediction error of X by multiple step:', rrmse(U_pred_multi_step,U)) 

The prediction error of Z by multiple step: 0.002415704866500827
The prediction error of X by multiple step: 0.0026152199138098136


# Parameter-to-coefficient mapping

In [94]:
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.interpolate import Rbf

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [95]:
Para_data = np.hstack((Para.reshape(-1,1),coef_matrix))

## RBF

In [96]:
X = Para_data[:, 0:1].reshape(-1,1)        
Y = Para_data[:, 1:] 
N, r = Y.shape

rbf_models = []
for j in range(r):
    rbf = Rbf(X[:,0], Y[:, j], function='multiquadric')
    rbf_models.append(rbf)

## NN

In [107]:
X = Para_data[:, 0:1].reshape(-1,1)        
Y = Para_data[:, 1:]       

X_train_t = torch.from_numpy(X).float().to(device)
Y_train_t = torch.from_numpy(Y).float().to(device)
output_dim = Y.shape[1]

In [108]:
class FourNet_sin(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, 128)
        self.linear2 = nn.Linear(128, 64)
        self.linear3 = nn.Linear(64,64)
        self.linear4 = nn.Linear(64,output_dim)

    def forward(self, x):
        x = torch.sin(self.linear1(x))
        x = torch.sin(self.linear2(x))
        x = torch.sin(self.linear3(x))
        x = self.linear4(x)
        return x

model = FourNet_sin(input_dim=1, output_dim=output_dim).to(device)

In [109]:
N, r = Y.shape
NN_models = []
for j in range(r):
    print(j)
    model = FourNet_sin(input_dim=1, output_dim=1).to(device)
    check_loss,running_time,Loss = train_fun(model,X_train_t,Y_train_t[:, j],N_red_lr=4,epochs=3000,lr=0.001,threshold=0.0001,printfun=False)
    NN_models.append(model)

0
运行时间： 5.956196069717407
loss: 9.4083727162797e-05
1
运行时间： 8.81092643737793
loss: 0.00017858555656857789
2
运行时间： 8.756361246109009
loss: 0.02657390385866165
3
运行时间： 11.411449909210205
loss: 0.010793172754347324
4
运行时间： 10.678783178329468
loss: 0.001054938300512731
5
运行时间： 11.354715824127197
loss: 4.0255170461023226e-05
6
运行时间： 7.313312768936157
loss: 0.0015400632983073592
7
运行时间： 13.165222883224487
loss: 0.000681428296957165
8
运行时间： 11.976621389389038
loss: 0.01539659220725298
9
运行时间： 4.451494932174683
loss: 0.014844872988760471
10
运行时间： 14.725659370422363
loss: 0.0013547976268455386
11
运行时间： 10.4271821975708
loss: 3.5330314858583733e-05
12
运行时间： 12.228593111038208
loss: 0.0013324219034984708
13
运行时间： 10.105772018432617
loss: 0.0002452115004416555
14
运行时间： 6.685502767562866
loss: 0.0016266308957710862
15
运行时间： 5.952584266662598
loss: 0.00281078671105206
16
运行时间： 4.38087010383606
loss: 0.023634914308786392
17
运行时间： 7.042792320251465
loss: 0.01888304576277733
18
运行时间： 3.187574625015259


# Online predict

## Set test parameter point

In [99]:
Para_test = np.array([0.654])
Para_test_number = len(Para_test)

U_test_all = []

for point in Para_test:
    U_alpha,V_alpha = RD_data(u0,v0,point,Lx,Nx,Ly,Ny,0.001,4)
    U_alpha = U_alpha[::10,:]
    U_test_all.append(U_alpha)

U_test = np.vstack(U_test_all)  

In [101]:
# get latent initial condiction
U_centered_test = U_test - U_mean
U_snapshots_test = U_centered_test.T 
Z_test = np.dot(Phi_r.T, U_snapshots_test).T

## RBF

In [102]:
# predcit the coefficients
Y_test_pred_RBF = []
for j, rbf in enumerate(rbf_models):
    Y_test_pred_RBF.append(rbf(Para_test.reshape(-1,1)))
Y_test_pred_RBF = np.hstack(Y_test_pred_RBF)

# sloving the latent dynamical system 
Library = []
for j in range(Para_test_number):   
    f, info = build_f_sparse_mixed(feature_names , Supports=Supports, 
                              coef_matrix=Y_test_pred_RBF[j].reshape(latent_dim,-1), latent_dim=latent_dim)
    z0 =  Z_test[m*j,:]
    T, Y = solve_rk4_adaptive_fixed_dt(f, 0, dt*(m-1), z0, dt, rtol=1e-6, atol=1e-9, h0=1e-2, h_min=0.001)
    Library.append(Y)
Z_pred_multi_step_RBF = np.vstack(Library)

# reconstruction 
U_pred_multi_step_RBF = np.dot(Z_pred_multi_step_RBF, Phi_r.T) + U_mean  # shape = (m, n)

In [103]:
rrmse(U_pred_multi_step_RBF,U_test)

0.004652910721651121

## NN

In [112]:
# predcit the coefficients
X_test = Para_test.reshape(-1,1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_test_t = torch.from_numpy(X_test).float().to(device)
Y_test_pred_NN = []
model.eval()
with torch.no_grad():
    for j, model in enumerate(NN_models):
        Y_test_pred_NN.append(model(X_test_t).cpu().numpy().reshape(-1,1))
Y_test_pred_NN = np.hstack(Y_test_pred_NN)

# sloving the latent dynamical system 
Library = []
for j in range(Para_test_number):   
    f, info = build_f_sparse_mixed(feature_names , Supports=Supports, 
                              coef_matrix=Y_test_pred_NN[j].reshape(latent_dim,-1), latent_dim=latent_dim)
    z0 =  Z_test[m*j,:]
    T, Y = solve_rk4_adaptive_fixed_dt(f, 0, dt*(m-1), z0, dt, rtol=1e-6, atol=1e-9, h0=1e-2, h_min=0.001)
    Library.append(Y)
Z_pred_multi_step_NN = np.vstack(Library)

# reconstruction 
U_pred_multi_step_NN= np.dot(Z_pred_multi_step_NN, Phi_r.T) + U_mean  # shape = (m, n)

In [113]:
rrmse(U_pred_multi_step_NN,U_test)

0.003368879149008316